In [1]:
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.6 MB/s eta 0:00:00a 0:00:01


In [2]:
import os, yaml, shutil
from ultralytics import YOLO

# ── Configuration ──────────────────────────────────────────────
DATASET_PATH = '/kaggle/input/datasets/abhishek162kumar/seatbelt-detection-dataset/Cleaned_Dataset_Seatbelt_Detection'  # <-- CHANGE THIS
PROJECT_NAME = 'seatbelt_detection'
MODEL_BASE   = 'yolov8n.pt'
EPOCHS       = 60
IMG_SIZE     = 640
BATCH_SIZE   = 16
# ───────────────────────────────────────────────────────────────

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
import yaml

config = {
    'path': '/kaggle/input/datasets/abhishek162kumar/seatbelt-detection-dataset/Cleaned_Dataset_Seatbelt_Detection',
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'names': ['Seatbelt', 'NoSeatbelt']
}

with open('/kaggle/working/data.yaml', 'w') as f:
    yaml.dump(config, f)

!ls /kaggle/working

data.yaml


In [4]:
data_yaml = '/kaggle/working/data.yaml'

model = YOLO(MODEL_BASE)

results = model.train(
    data    = data_yaml,
    epochs  = EPOCHS,
    imgsz   = IMG_SIZE,
    batch   = BATCH_SIZE,
    project = '/kaggle/working/runs',
    name    = PROJECT_NAME,
    device  = 0,
    patience= 20,
    save    = True,
    plots   = True,
    # Seatbelt detection benefits from close-crop augmentation
    scale   = 0.5,
    translate = 0.1,
    degrees = 5.0,
    fliplr  = 0.5,
    mosaic  = 0.5,   # lower mosaic — seatbelt is a small region
    copy_paste = 0.1,
)

Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=seatbelt_detection, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patienc

In [5]:
best_weights = f'/kaggle/working/runs/{PROJECT_NAME}/weights/best.pt'
model_best = YOLO(best_weights)
metrics = model_best.val(data=data_yaml, imgsz=IMG_SIZE)
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)

Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 1.0±1.4 ms, read: 29.3±18.9 MB/s, size: 39.1 KB)
val: Scanning /kaggle/input/datasets/abhishek162kumar/seatbelt-detection-dataset/Cleaned_Dataset_Seatbelt_Detection/valid/labels... 1060 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1060/1060 370.5it/s 2.9s0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/abhishek162kumar/seatbelt-detection-dataset/Cleaned_Dataset_Seatbelt_Detection/valid is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 67/67 7.3it/s 9.1s0.1s
                   all       1060       1487      0.918      0.904      0.946      0.625
              Seatbelt        655        835       0.92      0.898       0.94      0.623
            NoSeatbelt        532        652      0.917

In [6]:
output_path = '/kaggle/working/model2_seatbelt.pt'
shutil.copy(best_weights, output_path)
print(f'Model saved: {output_path}')

Model saved: /kaggle/working/model2_seatbelt.pt
